## Passos para RNN
1. Prepare and Load Data
2. Define network architecture
3. Define Loss and optimizer
4. Run the training loop
5. Evaluate the model


In [129]:
import torch
from torch.utils.data import DataLoader, random_split
from datasets import load_dataset
import torch.nn as nn
import torch.optim as optim

In [130]:
import re
from collections import Counter
from torch.nn.utils.rnn import pad_sequence

# The HF Hub now requires 'namespace/name' ids, so plain "imdb" fails
raw_dataset = load_dataset("stanfordnlp/imdb")  # splits: train (25k), test (25k), unsupervised

# Split the original train set into train/validation; keep the official test set
generator = torch.Generator().manual_seed(42)
val_ratio = 0.2
full_train = raw_dataset["train"]
val_size = int(len(full_train) * val_ratio)
train_size = len(full_train) - val_size
train_subset, val_subset = random_split(full_train, [train_size, val_size], generator=generator)
test_subset = raw_dataset["test"]

def tokenize(text):
	text = text.lower().replace("<br />", " ")
	return re.findall(r"[a-z0-9']+", text)

# Vocabulary built only from the training split to avoid leakage
max_vocab_size = 20000
counter = Counter()
for example in train_subset:
	counter.update(tokenize(example["text"]))
PAD_IDX, UNK_IDX = 0, 1
itos = ["<pad>", "<unk>"] + [word for word, _ in counter.most_common(max_vocab_size)]
vocab = {word: idx for idx, word in enumerate(itos)}

def encode(text, max_len=256):
	return [vocab.get(token, UNK_IDX) for token in tokenize(text)[:max_len]]

def collate_batch(batch):
	sequences = [torch.tensor(encode(example["text"]), dtype=torch.long) for example in batch]
	lengths = torch.tensor([len(seq) for seq in sequences])
	labels = torch.tensor([example["label"] for example in batch], dtype=torch.long)
	padded = pad_sequence(sequences, batch_first=True, padding_value=PAD_IDX)
	return padded, lengths, labels

batch_size = 64
train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True, collate_fn=collate_batch)
validation_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False, collate_fn=collate_batch)
test_loader = DataLoader(test_subset, batch_size=batch_size, shuffle=False, collate_fn=collate_batch)

print(f"Train: {len(train_subset)} | Validation: {len(val_subset)} | Test: {len(test_subset)} | Vocab: {len(vocab)}")

Train: 20000 | Validation: 5000 | Test: 25000 | Vocab: 20002


In [ ]:
class RNN(nn.Module):
	def __init__(self, input_size, hidden_size):
		super().__init__()
		self.hidden_size = hidden_size
		self.U = nn.Linear(input_size, hidden_size)
		self.W = nn.Linear(hidden_size, hidden_size, bias=False)

	def forward(self, x, h_prev):
		# choose tanh as activation function
		return torch.tanh(self.U(x) + self.W(h_prev))

	def init_hidden(self, batch_size):
		return self.U.weight.new_zeros(batch_size, self.hidden_size)


class Encoder(nn.Module):
	def __init__(self, vocab_size, embedding_size, hidden_size, padding_idx):
		super().__init__()
		self.embedding = nn.Embedding(vocab_size, embedding_size, padding_idx=padding_idx)
		self.rnn = RNN(embedding_size, hidden_size)

	def forward(self, tokens, lengths):
		embedded = self.embedding(tokens)
		hidden = self.rnn.init_hidden(tokens.size(0))
		lengths = lengths.to(tokens.device)
		for timestep in range(tokens.size(1)):
			next_hidden = self.rnn(embedded[:, timestep], hidden)
			active = (timestep < lengths).unsqueeze(1)
			hidden = torch.where(active, next_hidden, hidden)
		return hidden


class Decoder(nn.Module):
	def __init__(self, hidden_size, output_size):
		super().__init__()
		self.classifier = nn.Linear(hidden_size, output_size)

	def forward(self, context):
		return self.classifier(context)


class EncoderDecoder(nn.Module):
	def __init__(self, vocab_size, embedding_size, hidden_size, output_size, padding_idx):
		super().__init__()
		self.encoder = Encoder(vocab_size, embedding_size, hidden_size, padding_idx)
		self.decoder = Decoder(hidden_size, output_size)

	def forward(self, tokens, lengths):
		context = self.encoder(tokens, lengths)
		return self.decoder(context)

In [ ]:
def check_encoder_decoder():
	with torch.random.fork_rng(devices=[]):
		torch.manual_seed(42)
		check_model = EncoderDecoder(20, 8, 12, 2, padding_idx=0)
		tokens = torch.tensor([[2, 3, 4], [5, 0, 0], [0, 0, 0]])
		lengths = torch.tensor([3, 1, 0])
		logits = check_model(tokens, lengths)
		assert logits.shape == (3, 2)
		extra_padding = torch.zeros(3, 4, dtype=torch.long)
		torch.testing.assert_close(
			logits, check_model(torch.cat([tokens, extra_padding], dim=1), lengths)
		)
		torch.testing.assert_close(
			logits[1], check_model(torch.tensor([[5]]), torch.tensor([1]))[0]
		)
		empty_logits = check_model(torch.empty(1, 0, dtype=torch.long), torch.tensor([0]))
		torch.testing.assert_close(logits[2], empty_logits[0])
		loss = nn.CrossEntropyLoss()(logits, torch.tensor([1, 0, 1]))
		loss.backward()
		for parameter in check_model.parameters():
			assert parameter.grad is not None
			assert torch.isfinite(parameter.grad).all()
		assert check_model.encoder.embedding.weight.grad[0].eq(0).all()
		optim.Adam(check_model.parameters(), lr=0.001).step()
	print("Passed: variable lengths, padding invariance, empty reviews, and backward/optimizer step.")


check_encoder_decoder()

719


In [134]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
embedding_size = 128
hidden_size = 128
output_size = 2
model = EncoderDecoder(
	vocab_size=len(vocab),
	embedding_size=embedding_size,
	hidden_size=hidden_size,
	output_size=output_size,
	padding_idx=PAD_IDX,
).to(device)
print(f"Training on {device}")

Training on cuda


In [135]:
criterion = nn.CrossEntropyLoss()
learning_rate = 0.001
optimizer = optim.Adam(model.parameters(), learning_rate)

In [136]:
def train(model, loader, criterion, optimizer, device):
	model.train()
	total_loss = 0.0
	total_correct = 0
	total_examples = 0
	for tokens, lengths, labels in loader:
		tokens, labels = tokens.to(device), labels.to(device)
		optimizer.zero_grad()
		logits = model(tokens, lengths)
		loss = criterion(logits, labels)
		loss.backward()
		nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
		optimizer.step()
		total_loss += loss.item() * labels.size(0)
		total_correct += (logits.argmax(dim=1) == labels).sum().item()
		total_examples += labels.size(0)
	return total_loss / total_examples, total_correct / total_examples


@torch.no_grad()
def evaluate(model, loader, criterion, device):
	model.eval()
	total_loss = 0.0
	total_correct = 0
	total_examples = 0
	for tokens, lengths, labels in loader:
		tokens, labels = tokens.to(device), labels.to(device)
		logits = model(tokens, lengths)
		loss = criterion(logits, labels)
		total_loss += loss.item() * labels.size(0)
		total_correct += (logits.argmax(dim=1) == labels).sum().item()
		total_examples += labels.size(0)
	return total_loss / total_examples, total_correct / total_examples

In [137]:
num_epochs = 5
for epoch in range(num_epochs):
	train_loss, train_accuracy = train(model, train_loader, criterion, optimizer, device)
	val_loss, val_accuracy = evaluate(model, validation_loader, criterion, device)
	print(
		f"Epoch {epoch + 1}/{num_epochs} | "
		f"Train loss: {train_loss:.4f}, accuracy: {train_accuracy:.2%} | "
		f"Validation loss: {val_loss:.4f}, accuracy: {val_accuracy:.2%}"
	)

test_loss, test_accuracy = evaluate(model, test_loader, criterion, device)
print(f"Test loss: {test_loss:.4f}, accuracy: {test_accuracy:.2%}")

Epoch 1/5 | Train loss: 0.6627, accuracy: 59.65% | Validation loss: 0.6334, accuracy: 64.66%
Epoch 2/5 | Train loss: 0.5821, accuracy: 70.06% | Validation loss: 0.5793, accuracy: 70.76%
Epoch 3/5 | Train loss: 0.5114, accuracy: 76.10% | Validation loss: 0.5768, accuracy: 70.52%
Epoch 4/5 | Train loss: 0.4621, accuracy: 79.14% | Validation loss: 0.5283, accuracy: 75.90%
Epoch 5/5 | Train loss: 0.4123, accuracy: 82.24% | Validation loss: 0.5721, accuracy: 71.16%
Test loss: 0.5704, accuracy: 71.03%


In [ ]:
@torch.inference_mode()
def predict_sentiment(model, text):
	if not isinstance(text, str):
		raise TypeError("text must be a string.")
	encoded_text = encode(text)
	if not encoded_text:
		raise ValueError("text must contain at least one token.")

	model_device = next(model.parameters()).device
	tokens = torch.tensor([encoded_text], dtype=torch.long, device=model_device)
	lengths = torch.tensor([len(encoded_text)], dtype=torch.long, device=model_device)
	was_training = model.training
	model.eval()
	try:
		probabilities = model(tokens, lengths).softmax(dim=1)[0]
		prediction = probabilities.argmax().item()
		sentiments = ("negative", "positive")
		return {
			"label": sentiments[prediction],
			"confidence": probabilities[prediction].item(),
			"probabilities": {
				sentiment: probability
				for sentiment, probability in zip(sentiments, probabilities.tolist())
			},
		}
	finally:
		model.train(was_training)


text = test_loader.dataset[0]["text"]
result = predict_sentiment(model, text)
print(result)
print("Actual label:", "positive" if test_loader.dataset[0]["label"] == 1 else "negative")

AttributeError: 'list' object has no attribute 'lower'